# DSCI 552 — Homework 4
**Seonmin Kim**

Time Series Classification Part 2: Binary and Multiclass Classification (AReM), plus ISLR 4.8.3, 4.8.7.

## 0. Setup — HW3 코드 재사용 (데이터 로딩 / train-test split / 특징 추출)
HW4 지시문: HW3의 feature creation 코드가 필요하므로 그대로 포함.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/AReM")

COLS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

def load_instance(path):
    df = pd.read_csv(path, comment="#", header=None,
                     sep=r"[,\s]+", engine="python",
                     names=range(8))
    df = df.dropna(axis=1, how="all")
    df = df.iloc[:, 1:7]
    df.columns = COLS[1:]
    return df.astype(float)

ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]

instances = []
for act in ACTIVITIES:
    for f in sorted((DATA_DIR / act).glob("*.csv")):
        num = int(f.stem.replace("dataset", ""))
        instances.append({"activity": act, "num": num, "data": load_instance(f)})

print(len(instances))

In [ ]:
def is_test(inst):
    if inst["activity"] in ("bending1", "bending2"):
        return inst["num"] in (1, 2)
    return inst["num"] in (1, 2, 3)

test = [i for i in instances if is_test(i)]
train = [i for i in instances if not is_test(i)]
print(len(train), len(test))

In [ ]:
def extract_features(ts_df):
    out = {}
    for k, col in enumerate(ts_df.columns, start=1):
        s = ts_df[col]
        out[f"min{k}"]       = s.min()
        out[f"max{k}"]       = s.max()
        out[f"mean{k}"]      = s.mean()
        out[f"median{k}"]    = s.median()
        out[f"std{k}"]       = s.std()
        out[f"1st_quart{k}"] = s.quantile(0.25)
        out[f"3rd_quart{k}"] = s.quantile(0.75)
    return out

## 2(a)i 산점도 — bending vs 기타 (시계열 1, 2, 6)
**힌트**
- 1(c)iv에서 고른 3개 특징(예: min, mean, max)을 시계열 1, 2, 6에서 뽑는다 → 특징 9개.
- train 인스턴스만 사용. `label = (activity가 bending1/bending2)`로 이진 라벨.
- `seaborn.pairplot(df, hue="bending")` 하나면 끝. (교재 p.129 스타일)

In [ ]:
# TODO: 직접 작성


## 2(a)ii 시계열을 2등분 후 같은 산점도
**힌트**
- 각 시계열 480개 → 앞 240 / 뒤 240으로 쪼개면 인스턴스당 12개 시계열.
- `np.array_split(s, 2)` 를 쓰면 l을 일반화할 때 그대로 재사용 가능.
- 쪼갠 뒤 같은 특징을 뽑아 pairplot을 다시 그리고, 2(a)i와 차이가 있는지 한 줄로 서술.

In [ ]:
# TODO: 직접 작성


## 2(a)iii l ∈ {1,…,20} 분할 + 로지스틱 회귀 + RFE, 5-fold Stratified CV로 (l, p) 선택
**힌트 (구조)**
1. `build_features(inst_list, l)` 함수: 각 시계열을 `np.array_split`으로 l등분 → 조각마다 7개 특징 → 특징 수 = 6×l×7.
2. 바깥 루프 `for l in range(1, 21)`, 안쪽에서 `RFECV` 또는 `for p in ...` + `cross_val_score`.
3. **CV의 올바른 방법**: 특징 선택(RFE)을 각 fold의 train 부분 안에서만 수행해야 한다. 전체 데이터로 RFE를 먼저 하고 CV를 돌리면 정보 누수 → 이게 "wrong way". `Pipeline([("rfe", RFE(...)), ("lr", LogisticRegression(...))])`을 `cross_val_score`에 넣으면 자동으로 올바르게 된다.
4. 클래스 불균형(bending 9 vs 기타 60) → `StratifiedKFold(5)`.
5. L2 효과 제거: `LogisticRegression(C=1e6, max_iter=...)` (각주 3, 5 참고).
6. p-value가 필요하면 `statsmodels.api.Logit` 사용 (최종 모델 보고용).

In [ ]:
# TODO: 직접 작성


## 2(a)iv 최적 모델 — train confusion matrix, ROC, AUC, β와 p-value
**힌트**
- `sklearn.metrics.confusion_matrix`, `RocCurveDisplay.from_estimator`(또는 `roc_curve`+`auc`).
- β와 p-value는 statsmodels `Logit(y, sm.add_constant(X_selected)).fit()` 결과의 `.params`, `.pvalues`.
- 완전 분리(perfect separation)로 수렴 경고가 나올 수 있음 — 그 자체가 2(a)vi의 답 재료.

In [ ]:
# TODO: 직접 작성


## 2(a)v 테스트셋 평가
**힌트**
- 테스트 시계열도 **같은 l로** 쪼개서 같은 특징을 추출해야 한다.
- test accuracy vs CV accuracy 비교 한 줄.

In [ ]:
# TODO: 직접 작성


## 2(a)vi 클래스가 잘 분리되어 불안정성이 생기는가? (서술)

_답 작성_

## 2(a)vii 불균형 → case-control sampling
**힌트**
- bending 9개 vs 기타 60개 → 불균형 맞음.
- case-control: 다수 클래스를 소수 클래스 수만큼 다운샘플링해 학습 → 절편 보정 β₀ + log(π/(1−π) × n_other/n_bend) (ISLR 4.6.3 / 강의 노트의 보정식).
- 보정 후 confusion matrix, ROC, AUC 다시 보고.

In [ ]:
# TODO: 직접 작성


## 2(b)i L1-penalized 로지스틱 회귀 — (l, λ) 교차검증
**힌트**
- `LogisticRegressionCV(penalty="l1", solver="liblinear", Cs=..., cv=StratifiedKFold(5), scoring="accuracy")` — λ(C) CV는 자동.
- l은 여전히 바깥 루프. 특징은 표준화 권장 → `Pipeline([("sc", StandardScaler()), ("lr", LogisticRegressionCV(...))])`.

In [ ]:
# TODO: 직접 작성


## 2(b)ii p-value 변수선택 vs L1 비교 (서술)
성능(정확도)과 구현 난이도 두 축으로 비교.

_답 작성_

## 2(c)i L1 멀티노미얼 회귀 — 7개 활동 전부 분류
**힌트**
- 라벨 = activity (7클래스). l은 2(b)i와 같은 방식으로 CV.
- `LogisticRegression(penalty="l1", solver="saga", max_iter=...)` (multinomial은 saga만 L1 지원).
- test error 보고. 멀티클래스 confusion matrix는 `ConfusionMatrixDisplay`, ROC는 one-vs-rest로 클래스별 곡선.

In [ ]:
# TODO: 직접 작성


## 2(c)ii 나이브 베이즈 — Gaussian vs Multinomial
**힌트**
- `GaussianNB()`, `MultinomialNB()` — Multinomial은 음수 특징 불가 → `MinMaxScaler`로 0~1 스케일링.
- 같은 l, 같은 특징으로 test error 비교.

In [ ]:
# TODO: 직접 작성


## 2(c)iii 어느 방법이 더 좋은가? (서술)

_답 작성_

## 3. ISLR 4.8.3 (서술)
QDA에서 클래스별 분산이 다를 때 베이즈 분류기가 2차(quadratic)임을 보이는 문제. 4.4.2절의 판별함수 유도에서 σ²에 아래첨자 k를 남기고 전개.

_답 작성_

## 4. ISLR 4.8.7 (서술)
배당 문제 — 베이즈 정리로 P(Yes | X=4) 계산. 정규분포 밀도 두 개 대입: π_yes=0.8, μ_yes=10, μ_no=0, σ²=36.

_답 작성_

---
## AI 사용 공개 (AI Disclosure)
Claude를 사용해 레포 구조 세팅, HW3 코드 재사용 정리, 각 문항 접근 방향 힌트를 받았으며, 모든 모델링 코드와 답안은 직접 작성함.